#  <span style="color:royalblue">Проект: системы обработки больших данных

<div style="background-color: white; border: 2px solid yellowgreen; border-radius: 5px; padding: 15px 25px; display: inline-block; color: black;">

---

## Описание проекта

---
    
<span style="color:royalblue">

Ц Е Л Ь
- <span style="color:black"> Обучить модель линейной регрессии для предсказания медианной стоимости недвижимости в Калифорнии в 1990 году, используя библиотеку машинного обучения Apache Spark MLlib.

З А Д А Ч И
    
**1.**  <span style="color:black">Инициализировать локальную Spark-сессию.

**2.** <span style="color:black">Обучить две модели линейной регрессии (с использованием `LinearRegression` из `pyspark.ml.regression`) на подготовленных наборах данных.

**3.** <span style="color:black">Оценить качество обеих моделей на тестовой выборке с помощью метрик `RMSE, MAE и R2`, и сделать вывод о влиянии категориального признака на точность предсказаний.

---

ОПИСАНИЕ ДАННЫХ

<span style="color:black">и с т о ч н и к:

- <span style="color:black">В проекте используется датасет, содержащий информацию о жилых массивах (блоках) Калифорнии за 1990 год.

<span style="color:black">п р и з н а к и:

- <span style="color:black">Данные представлены в файле `/datasets/housing.csv` и содержат следующие поля:

| название колонки | описание |
| :--- | :--- | 
| `longitude` | Географическая долгота расположения блока. |
| `latitude` | Географическая широта расположения блока. |
| `housing_median_age` | Медианный возраст жителей жилого массива. |
| `total_rooms` | Общее количество комнат в домах жилого массива. |
| `total_bedrooms` | Общее количество спален в домах жилого массива. |
| `population` | Количество человек, которые проживают в жилом массиве. |
| `households` | Количество домовладений в жилом массиве. |
| `median_income` | медианный доход жителей жилого массива. |
| `median_house_value` | **Целевая переменная.** Медианная стоимость дома в жилом массиве. |
| `ocean_proximity` | **Категориальный признак.** Характеризует близость блока к океану (например, "<1H OCEAN", "INLAND", "NEAR BAY" и т.д.). |


## Импорт данных

In [1]:
!pip install pyspark

     |████████████████████████████████| 198 kB 2.7 MB/s eta 0:00:01
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.3
    Uninstalling py4j-0.10.9.3:
      Successfully uninstalled py4j-0.10.9.3


In [2]:
# Стандартные библиотеки Python
import os
import sys

# Научные библиотеки
import numpy as np
import pandas as pd

# PySpark основные
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, isnan, isnull

# PySpark ML
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer  # для заполнения пропусков
from pyspark.ml.feature import StringIndexer, StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression

# Версионно-зависимые импорты
pyspark_version = pyspark.__version__
if int(pyspark_version[:1]) == 3:
    from pyspark.ml.feature import OneHotEncoder    
elif int(pyspark_version[:1]) == 2:
    from pyspark.ml.feature import OneHotEncoderEstimator as OneHotEncoder
        
# Константы
RANDOM_SEED = 2022

In [3]:
# Устанавливаем переменные окружения ДО создания SparkSession
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Также добавляем путь к Python в PATH
python_path = os.path.dirname(sys.executable)
if python_path not in os.environ['PATH']:
    os.environ['PATH'] = python_path + ';' + os.environ['PATH']

In [4]:
# Инициализация Spark-сессии
spark = SparkSession.builder \
                    .master("local") \
                    .appName("EDA California Housing") \
                    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [5]:
# Список возможных путей к файлу
possible_paths = [
                  'housing.csv',
                  './housing.csv',
                  '../housing.csv',
                  '/datasets/housing.csv',
                  'C:\\datasets\\housing.csv',  # Windows
                  '/content/housing.csv',        # Google Colab
                  '/home/user/datasets/housing.csv',  # Linux
                 ]

file_found = False
for path in possible_paths:
    if os.path.exists(path):
        print(f"Файл найден: {path}")
        pandas_df = pd.read_csv(path)
        file_found = True
        break

if not file_found:
    print("Файл не найден локально. Загружаем из интернета...")
    url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
    pandas_df = pd.read_csv(url)
    # Сохраняем локально для будущих запусков
    pandas_df.to_csv('housing.csv', index=False)
    print("Файл сохранен как 'housing.csv'")

# Конвертируем в Spark DataFrame
df_housing = spark.createDataFrame(pandas_df)
print(f"Данные загружены, размер: {df_housing.count()} строк")

Файл найден: /datasets/housing.csv


Данные загружены, размер: 20640 строк


## Подготовка данных

<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">

<span style="color:royalblue">
    
- <span style="color:black">Данные были собраны в рамках переписи населения в США. Каждая строка содержит агрегированную статистику о жилом массиве.
- <span style="color:black">Жилой массив — минимальная географическая единица с населением от 600 до 3000 человек в зависимости от штата. 
- <span style="color:black">Одна строка в данных содержит статистику в среднем о 1425.5 обитателях жилого массива.

In [6]:
# выводим названия колонок и тип
pd.DataFrame(df_housing.dtypes, columns=['column', 'type']).head(10)

,column,type
0,longitude,double
1,latitude,double
2,housing_median_age,double
3,total_rooms,double
4,total_bedrooms,double
5,population,double
6,households,double
7,median_income,double
8,median_house_value,double
9,ocean_proximity,string


In [7]:
# выведем первые 5 строк
df_housing.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">

<span style="color:royalblue">
    
- <span style="color:black"> Вывели названия колонок и их тип данных в виде таблицы, используя атрибут dtypes.
- <span style="color:black"> Представили результат в виде таблицы pandas.
- <span style="color:black"> Выведите первые 10 строк датасета методом DataFrame API.

In [8]:
# выводим базовые статистики
df_housing.describe().toPandas()

,summary,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,count,20640,20640,20640,20640,20640,20640,20640,20640,20640,20640
1,mean,-119.56970445736148,35.6318614341087,28.639486434108527,2635.7630813953488,NaN,1425.4767441860465,499.5396802325581,3.8706710029070246,206855.81690891474,None
2,stddev,2.003531723502584,2.135952397457101,12.58555761211163,2181.6152515827944,NaN,1132.46212176534,382.3297528316098,1.899821717945263,115395.61587441359,None
3,min,-124.35,32.54,1.0,2.0,1.0,3.0,1.0,0.4999,14999.0,<1H OCEAN
4,max,-114.31,41.95,52.0,39320.0,NaN,35682.0,6082.0,15.0001,500001.0,NEAR OCEAN


<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">

<span style="color:royalblue">
    
- <span style="color:black"> В датасете 20640 строк.
- <span style="color:black"> Максимальная медианная стоимость дома `median_house_value` равна 500001 долларов.
- <span style="color:black"> Видим пропущенные значения в столбце `total_bedrooms`. Далее посчитаем кол-во пропусков и обработаем их.

### Обработка пропущенных значений

<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">
    
<span style="color:royalblue">
    
- <span style="color:black"> Найдём количество пропусков в каждой колонке.

In [9]:
nan = df_housing.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in df_housing.columns]).toPandas().T.rename(columns={0:'NA'})
nan['NA_percents'] = nan['NA'] / df_housing.count() * 100
display(nan)

,NA,NA_percents
longitude,0,0.000000
latitude,0,0.000000
housing_median_age,0,0.000000
total_rooms,0,0.000000
total_bedrooms,207,1.002907
population,0,0.000000
households,0,0.000000
median_income,0,0.000000
median_house_value,0,0.000000
ocean_proximity,0,0.000000


<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">
    
<span style="color:royalblue">
    
- <span style="color:black">В столбце `total_bedrooms` были обнаружены пропуски.
- <span style="color:black">В нашем датасете 20640 строк. При этом пропуски есть только в 207 строках.
- <span style="color:black">Заполним пропуски медианным значением в пайплайне.

<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">

<span style="color:royalblue">
    
- <span style="color:black">Разделим признаки на **`категориальные`**, **`числовые`** и **`целевой признак`**.

In [11]:
# выделим признаки
num = ['longitude'
        ,'latitude'
        ,'housing_median_age'
        ,'total_rooms'
        ,'total_bedrooms'
        ,'population'
        ,'households'
        ,'median_income'
        ]
cat = ['ocean_proximity']
target = 'median_house_value'

## Разделение на выборки

<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">

<span style="color:royalblue">
    
- <span style="color:black">Разделяем наш датасет на две части — выборку для обучения и выборку для тестирования качества модели с помощью метода **`randomSplit()`**.

In [12]:
train_df, test_df = df_housing.randomSplit([0.8, 0.2], seed=RANDOM_SEED)

print(f"\nОбучающая выборка: {train_df.count()} строк")
print(f"Тестовая выборка: {test_df.count()} строк")


Обучающая выборка: 16418 строк
Тестовая выборка: 4222 строк


## Создаём Pipeline для всех трансформаций

In [15]:
# 1. Импутер для пропусков (заполняет медианой из train)
imputer = Imputer(inputCols=['total_bedrooms'], 
                  outputCols=['total_bedrooms_imputed'],
                  strategy='median')

# 2. Индексатор категорий
indexer = StringIndexer(inputCol='ocean_proximity', 
                        outputCol='ocean_proximity_idx',
                        handleInvalid='keep')

# 3. OneHotEncoder. Преобразуем колонку с категориальными значениями
encoder = OneHotEncoder(inputCol='ocean_proximity_idx', 
                        outputCol='ocean_proximity_ohe')

# Соберём трансформированные категорийные и числовые признаки с помощью VectorAssembler.

# 4. Ассемблер для числовых признаков. Oбъединеняем признаки в один вектор
num_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                'total_bedrooms_imputed', 'population', 'households', 'median_income']

numerical_assembler = VectorAssembler(inputCols=num_features, 
                                outputCol='numerical_features')

# 5. Стандартизация. Масштабируем колонки с численными значениями
scaler = StandardScaler(inputCol='numerical_features', 
                        outputCol='numerical_features_scaled',
                        withStd=True, withMean=True)

# 6. Финальный ассемблер для ВСЕХ признаков
all_assembler = VectorAssembler(inputCols=['ocean_proximity_ohe', 
                                          'numerical_features_scaled'],
                                outputCol='features')

In [16]:
#Pipeline для модели №1 (со всеми признаками)
pipeline_all = Pipeline(stages=[
                                imputer,
                                indexer,
                                encoder,
                                numerical_assembler,
                                scaler,
                                all_assembler
                                ])

In [17]:
#Pipeline для модели №2 (для численных признаков)
num_assembler_final = VectorAssembler(
                                     inputCols=['numerical_features_scaled'],
                                     outputCol='features'
                                      )

pipeline_num = Pipeline(stages=[
                               imputer,
                               numerical_assembler,
                               scaler,
                               num_assembler_final
                               ])

## Обучение модели

<div style="background-color: white; border: 1px solid crimson; border-radius: 0px; padding: 15px 25px; display: inline-block; color: black;">

Построим две модели линейной регрессии на разных наборах данных:

<span style="color:royalblue">
    
- <span style="color:black">используя все признаки `[Модель №1]`
- <span style="color:black">используя только числовые переменные, исключив категориальные `[Модель №2]`

<span style="color:black">Для построения модели используем оценщик **`LinearRegression`** из библиотеки **`MLlib`**.

### Модель №1

`[Используем все признаки]`

In [18]:
# Обучение Для модели со всеми признаками
pipeline_all_model = pipeline_all.fit(train_df)
train_all = pipeline_all_model.transform(train_df)
test_all = pipeline_all_model.transform(test_df)

In [24]:
# МОДЕЛЬ 1: Со всеми признаками
lr = LinearRegression(labelCol=target, featuresCol='features')
model_all = lr.fit(train_all)
predictions_all = model_all.transform(test_all)
trainingSummary_all = model_all.summary

print('\nМодель №1 со всеми признаками:')
print("RMSE: %0.2f" % trainingSummary_all.rootMeanSquaredError)
print('MAE: %0.2f' % trainingSummary_all.meanAbsoluteError)
print("r2: %0.2f" % trainingSummary_all.r2)


Модель №1 со всеми признаками:
RMSE: 68839.15
MAE: 49832.84
r2: 0.64


### Модель №2

`[Используем только численные признаки]`

In [19]:
# Для модели только с числовыми признаками
pipeline_num_model = pipeline_num.fit(train_df)
train_num = pipeline_num_model.transform(train_df)
test_num = pipeline_num_model.transform(test_df)

In [23]:
# МОДЕЛЬ 2: Только с числовыми признаками
model_num = lr.fit(train_num)
predictions_num = model_num.transform(test_num)
trainingSummary_num = model_num.summary

print('\nМодель №2 с численными признаками:')
print("RMSE: %0.2f" % trainingSummary_num.rootMeanSquaredError)
print('MAE: %0.2f' % trainingSummary_num.meanAbsoluteError)
print("r2: %0.2f" % trainingSummary_num.r2)


Модель с численными признаками:
RMSE: 69780.19
MAE: 50944.69
r2: 0.63


In [ ]:
# останавливаем spark-сессию
spark.stop()

<div style="background-color: white; border: 2px solid yellowgreen; border-radius: 5px; padding: 15px 25px; display: inline-block; color: black;">

## в ы в о д

---

Сравним результаты работы линейной регрессии на двух наборах данных по метрикам **`RMSE`**, **`MAE`** и **`R2`**.

---

**Модель 1** `[со всеми признаками (категориальные + числовые) показывает ЛУЧШИЕ результаты]`

<span style="color:royalblue">
    
- <span style="color:black">Категориальный признак `ocean_proximity` добавляет полезную информацию для предсказания стоимости жилья.

- <span style="color:black">RMSE: меньше на 941 (68.8K vs 69.8K) — ошибка меньше на 1.3%

- <span style="color:black">MAE: меньше на 1,112 (49.8K vs 50.9K) — ошибка меньше на 2.2%

- <span style="color:black">R2: больше на 0.01 (0.64 vs 0.63) — объясняет на 1% больше вариации цен

Добавление информации о близости к океану действительно улучшает качество прогноза, хотя улучшение небольшое (1-2%). Это подтверждает, что местоположение относительно океана — значимый, но не главный фактор в ценообразовании жилья Калифорнии.